### New Code

In [7]:
import os
import re
import sys
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from ase.io import read, write
from ase import Atoms
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

# ============================================
# ===== PORTABLE PATH & CONFIG SETTINGS =====
# ============================================
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # Jupyter Notebook fallback
    SCRIPT_DIR = Path.cwd()

# Automatically resolve paths relative to the project root
PROJECT_DIR = SCRIPT_DIR if (SCRIPT_DIR / "results").exists() else SCRIPT_DIR.parent
LOWER_ROOT  = PROJECT_DIR / "results" / "chain_splitted" / "lower"
OUT_ROOT    = PROJECT_DIR / "results" / "generated_cifs_2"

FOLDERS = [f"r{d}" for d in range(0, 341, 20)]   # r0, r20, ..., r340
ANGLES  = np.arange(0, 341, 20)                  # 0..340 by 20
X_DISPS = np.arange(0.0, 9.7, 1.2)               # 0.0..9.6 by 1.2

COM_MODE = 'mass'   # 'mass' or 'geom'
ROTATION_SIGN = -1  # opposite direction for partner rotation (+1 original, -1 opposite)
# ============================================


def is_jupyter():
    try:
        shell = get_ipython().__class__.__name__
        return shell == 'ZMQInteractiveShell'
    except NameError:
        return False


# ---------- math helpers ----------
def rot_x(theta_rad: float) -> np.ndarray:
    c, s = np.cos(theta_rad), np.sin(theta_rad)
    return np.array([[1.0, 0.0, 0.0],
                     [0.0,  c, -s],
                     [0.0,  s,  c]], dtype=float)


def rotate_about_x_through_center(coords: np.ndarray,
                                  center_cart: np.ndarray,
                                  theta_rad: float) -> np.ndarray:
    R = rot_x(theta_rad)
    return (coords - center_cart) @ R.T + center_cart


# ---------- PBC / torus-safe helpers ----------
def circular_mean_1d(u: np.ndarray) -> float:
    ang = 2 * np.pi * u
    z = np.exp(1j * ang).mean()
    m = np.angle(z)
    if m < 0:
        m += 2 * np.pi
    return m / (2 * np.pi)


def torus_center_frac(scaled: np.ndarray) -> np.ndarray:
    return np.array([circular_mean_1d(scaled[:, i]) for i in range(3)], dtype=float)


def unwrap_scaled_around_center(scaled: np.ndarray, center_frac: np.ndarray) -> np.ndarray:
    d = scaled - center_frac
    d -= np.round(d)            # (-0.5, 0.5]
    return center_frac + d


def scaled_to_cart(scaled: np.ndarray, cell3x3: np.ndarray) -> np.ndarray:
    return scaled @ cell3x3


def cart_to_scaled(cart: np.ndarray, cell3x3: np.ndarray) -> np.ndarray:
    inv = np.linalg.inv(cell3x3)
    return cart @ inv


def nearest_image_shift_for_centers(partner_center_frac: np.ndarray,
                                    base_center_frac: np.ndarray,
                                    cell3x3: np.ndarray) -> np.ndarray:
    df = partner_center_frac - base_center_frac
    k_frac = np.round(df)       # nearest integer triplet
    return k_frac @ cell3x3     # cartesian shift to subtract


# ---------- IO/helpers ----------
def parse_lower_deg(rname: str) -> int:
    m = re.fullmatch(r"r(\d+)", rname)
    if not m:
        raise ValueError(f"Invalid r-folder name: {rname}")
    return int(m.group(1))


def find_lower_cif_for_rname(lower_root: Path, rname: str) -> Path:
    rdir = lower_root / rname
    exact = rdir / f"{rname}_t0_t0_0_lower.cif"
    if exact.exists():
        return exact

    if rdir.exists():
        hits = sorted(rdir.glob("*_lower.cif"))
        if len(hits) == 1:
            return hits[0]
        elif len(hits) > 1:
            raise FileNotFoundError(
                f"Ambiguous: multiple *_lower.cif under {rdir}: {[h.name for h in hits]}"
            )

    hits_root = sorted(p for p in lower_root.glob("r*_*_lower.cif")
                       if p.name.startswith(f"{rname}_") and p.name.endswith("_lower.cif"))
    if len(hits_root) == 1:
        return hits_root[0]
    elif len(hits_root) > 1:
        raise FileNotFoundError(
            f"Ambiguous: multiple {rname}*_lower.cif directly under {lower_root}: {[p.name for p in hits_root]}"
        )

    hits_recursive = sorted(p for p in lower_root.rglob(f"{rname}*_lower.cif"))
    if len(hits_recursive) == 1:
        return hits_recursive[0]
    elif len(hits_recursive) > 1:
        raise FileNotFoundError(
            f"Ambiguous (recursive): multiple matches for {rname} under {lower_root}: "
            f"{[str(p.relative_to(lower_root)) for p in hits_recursive]}"
        )

    raise FileNotFoundError(f"No lower CIF found for {rname} under {lower_root}")


def format_t_folder(d: float) -> str:
    return "t0" if abs(d) < 1e-9 else f"t{d:.1f}"


def build_file_id(rname: str, disp: float, angle: int) -> str:
    t_folder = format_t_folder(disp)
    return f"{rname}/{t_folder}/{t_folder}_{angle}.cif"


def normalize_tm_gen_inplace(tm: pd.DataFrame) -> None:
    colmap = {c: c.lower() for c in tm.columns if c.lower() in ("x","y","z") and c != c.lower()}
    if colmap:
        tm.rename(columns=colmap, inplace=True)
    need = {"x","y","z"}
    if not need.issubset(set(tm.columns)):
        raise ValueError(f"tm_gen missing columns: {need - set(tm.columns)}")
    if "file_id" not in tm.columns:
        if "file_path" not in tm.columns:
            raise ValueError("tm_gen needs 'file_id' or 'file_path'.")
        tm["file_path"] = tm["file_path"].astype(str).str.replace("\\", "/", regex=False)
        tm["file_id"]   = tm["file_path"].str.replace(
            r".*?(data/|generated_cifs/|chain_splitted/)", "", regex=True
        )
        tm["file_id"] = tm["file_id"].str.replace(".cif", "", regex=False) + ".cif"
    tm["file_id"] = tm["file_id"].astype(str).str.strip().str.lstrip("/")
    tm.drop_duplicates(subset=["file_id"], keep="first", inplace=True)


def build_xyz_lookup(tm: pd.DataFrame) -> dict:
    # Work on a copy of dataframe to prevent mutation leaks
    tm_copy = tm.copy()
    normalize_tm_gen_inplace(tm_copy)
    return {
        fid: (float(x), float(y), float(z))
        for fid, x, y, z in tm_copy[["file_id","x","y","z"]].itertuples(index=False, name=None)
    }


# ---------- parallel worker ----------
def process_single_rname(rname, lower_root, out_root, fid2xyz):
    try:
        lower_cif = find_lower_cif_for_rname(lower_root, rname)
    except FileNotFoundError as e:
        return {"status": "error", "rname": rname, "error": str(e), "written": 0, "miss_xyz": 0}

    # Load base lower chain
    base_atoms = read(str(lower_cif))
    base_syms  = base_atoms.get_chemical_symbols()
    orig_cell  = base_atoms.cell.array.copy()
    base_pbc   = base_atoms.get_pbc()

    # Torus-aware center & unwrapped copy for math (with original cell)
    base_scaled_wrapped = base_atoms.get_scaled_positions(wrap=True)
    base_center_frac = torus_center_frac(base_scaled_wrapped)
    base_scaled_unwrapped = unwrap_scaled_around_center(base_scaled_wrapped, base_center_frac)
    base_pos_unwrapped = scaled_to_cart(base_scaled_unwrapped, orig_cell)

    # Pivot (cartesian) from unwrapped base
    if COM_MODE.lower() == 'mass':
        tmp = Atoms(symbols=base_syms, positions=base_pos_unwrapped,
                    cell=orig_cell, pbc=base_pbc)
        pivot_cart = tmp.get_center_of_mass()
    else:
        pivot_cart = base_pos_unwrapped.mean(axis=0)

    # Build partner-at-origin by counter-rotating to 0°
    lower_deg = parse_lower_deg(rname)
    partner_at_origin = rotate_about_x_through_center(
        base_pos_unwrapped, pivot_cart, theta_rad=np.radians(-lower_deg)
    )

    written = 0
    miss_xyz = 0

    for disp in X_DISPS:
        t_folder = format_t_folder(disp)
        out_dir = out_root / rname / t_folder
        out_dir.mkdir(parents=True, exist_ok=True)

        for angle in ANGLES:
            fid = build_file_id(rname, disp, int(angle))
            xyz = fid2xyz.get(fid)
            if xyz is None:
                miss_xyz += 1
                continue

            # Opposite-direction extra rotation
            theta = np.radians(ROTATION_SIGN * angle)
            partner_rot = rotate_about_x_through_center(
                partner_at_origin, pivot_cart, theta_rad=theta
            )

            # Translate by tm_gen (x,y,z) + extra +disp along x
            x, y, z = xyz
            #partner_final = partner_rot + np.array([x + disp, y, z], dtype=float)
            partner_final = partner_rot + np.array([x , y, z], dtype=float)
            
            # Snap partner to same image as base (still using original cell!)
            partner_center_frac = cart_to_scaled(partner_final.mean(axis=0), orig_cell)
            shift_cart = nearest_image_shift_for_centers(
                partner_center_frac, base_center_frac, orig_cell
            )
            partner_final -= shift_cart

            # --------- NOW ENLARGE CELL BY ADDING TO EXISTING LENGTHS ----------
            enlarged_cell = orig_cell.copy()

            # add 50 Å to the current b-vector length (keep direction)
            b_vec = enlarged_cell[1]
            b_len = np.linalg.norm(b_vec)
            if b_len > 1e-8:
                new_b_len = b_len + 50.0
                enlarged_cell[1] = b_vec * (new_b_len / b_len)
            else:
                enlarged_cell[1] = np.array([0.0, 50.0, 0.0])

            # add 50 Å to the current c-vector length (keep direction)
            c_vec = enlarged_cell[2]
            c_len = np.linalg.norm(c_vec)
            if c_len > 1e-8:
                new_c_len = c_len + 50.0
                enlarged_cell[2] = c_vec * (new_c_len / c_len)
            else:
                enlarged_cell[2] = np.array([0.0, 0.0, 50.0])

            # --------- WRAP BOTH PARTS INTO THE NEW CELL ----------
            # base: use the nice continuous unwrapped positions, then wrap into enlarged cell
            base_scaled_enlarged = cart_to_scaled(base_pos_unwrapped, enlarged_cell)
            base_scaled_enlarged = base_scaled_enlarged % 1.0
            base_pos_for_output = scaled_to_cart(base_scaled_enlarged, enlarged_cell)

            # partner: wrap into enlarged cell too
            partner_scaled_enlarged = cart_to_scaled(partner_final, enlarged_cell)
            partner_scaled_enlarged = partner_scaled_enlarged % 1.0
            partner_pos_for_output = scaled_to_cart(partner_scaled_enlarged, enlarged_cell)

            # merge
            merged_pos  = np.vstack([base_pos_for_output, partner_pos_for_output])
            merged_syms = np.array(base_syms + base_syms, dtype=object)

            merged = Atoms(symbols=list(merged_syms), positions=merged_pos)
            merged.set_cell(enlarged_cell, scale_atoms=False)
            merged.set_pbc(base_pbc)

            outfile = out_dir / f"{t_folder}_{int(angle)}.cif"
            write(str(outfile), merged)
            written += 1

    return {"status": "success", "rname": rname, "written": written, "miss_xyz": miss_xyz}


# ---------- main execution ----------
def main():
    # 1) Locate/Load tm_gen displacement data
    if 'df_results' in globals():
        print("[INFO] Using df_results from active notebook memory.")
        tm_gen_data = globals()['df_results'].copy()
    else:
        csv_path = PROJECT_DIR / "results" / "cif_layer_displacements.csv"
        if csv_path.exists():
            print(f"[INFO] Loading displacement data from CSV: {csv_path}")
            tm_gen_data = pd.read_csv(csv_path)
        else:
            raise FileNotFoundError(f"Missing displacement CSV file. Please run displacement calculations first.")

    fid2xyz = build_xyz_lookup(tm_gen_data)
    print(f"[INFO] Loaded {len(fid2xyz)} lookup entries.")

    # 2) Choose parallel executor based on environment
    if is_jupyter() and sys.platform.startswith("win"):
        Executor = ThreadPoolExecutor  # ThreadPool protects Windows Jupyter instances from pickling errors
        executor_type = "ThreadPoolExecutor"
    else:
        Executor = ProcessPoolExecutor
        executor_type = "ProcessPoolExecutor"

    num_workers = max(1, os.cpu_count() - 1)
    print(f"[INFO] Generating structures with {num_workers} workers ({executor_type})...")

    processed = written = miss_xyz = miss_base = 0
    errors = []

    with Executor(max_workers=num_workers) as executor:
        futures = {
            executor.submit(process_single_rname, rname, LOWER_ROOT, OUT_ROOT, fid2xyz): rname
            for rname in FOLDERS
        }

        with tqdm(total=len(FOLDERS), desc="Generating Bilayer CIFs", unit="folder") as pbar:
            for future in as_completed(futures):
                res = future.result()
                if res["status"] == "success":
                    processed += 1
                    written += res["written"]
                    miss_xyz += res["miss_xyz"]
                else:
                    miss_base += 1
                    errors.append((res["rname"], res["error"]))
                pbar.update(1)

    print("\n================ GENERATION SUMMARY ================")
    print(f"  Rotation folders processed successfully: {processed}")
    print(f"  Missing base structures                 : {miss_base}")
    print(f"  Missing coordinate lookups (skipped)    : {miss_xyz}")
    print(f"  Total Bilayer CIF files generated       : {written}")
    print(f"  Output directory                        : {OUT_ROOT}")
    print("====================================================")

    if errors:
        print("\n--- Processing Errors ---")
        for rname, err in errors:
            print(f"Folder {rname}: {err}")


if __name__ == "__main__":
    main()


[INFO] Loading displacement data from CSV: d:\New folder\project\results\cif_layer_displacements.csv
[INFO] Loaded 2916 lookup entries.
[INFO] Generating structures with 7 workers (ThreadPoolExecutor)...


Generating Bilayer CIFs: 100%|██████████| 18/18 [00:06<00:00,  2.83folder/s]


================ GENERATION SUMMARY ================
  Rotation folders processed successfully: 18
  Missing base structures                 : 0
  Missing coordinate lookups (skipped)    : 0
  Total Bilayer CIF files generated       : 2916
  Output directory                        : d:\New folder\project\results\generated_cifs_2


### Features

In [8]:
import os
import re
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm
from ase.io import read
from pathlib import Path
from scipy.spatial import cKDTree as KDTree
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

# ============================================
# ===== PORTABLE PATH & CONFIG SETTINGS =====
# ============================================
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # Jupyter Notebook fallback
    SCRIPT_DIR = Path.cwd()

# Automatically resolve paths relative to the project root
PROJECT_DIR = SCRIPT_DIR if (SCRIPT_DIR / "results").exists() else SCRIPT_DIR.parent
BASE_DIR    = PROJECT_DIR / "results" / "generated_cifs_2"
OUTPUT_DIR  = PROJECT_DIR / "results"
OUTPUT_CSV  = "cif_bounds_summary_generated2_features.csv"

# ----------------- PARAMETERS -------------------
# van der Waals radii (Å) and tolerance
rvw_H   = 1.20
rvw_O   = 1.53
rvw_err = 0.10
# ------------------------------------------------


def is_jupyter():
    try:
        shell = get_ipython().__class__.__name__
        return shell == 'ZMQInteractiveShell'
    except NameError:
        return False


def validate_structure(atoms):
    if len(atoms) == 0:
        raise ValueError("Structure has zero atoms.")
    if atoms.get_cell().volume <= 0:
        raise ValueError("Invalid or zero-volume cell.")
    if np.isnan(atoms.get_positions()).any():
        raise ValueError("NaN detected in atomic positions.")


# ---------- geometry helpers ----------
def wrap_positions_custom(positions, cell, pbc, center=(0.5, 0.5, 0.5)):
    """
    Consistent coordinate wrapping centered at 0.5 to keep layers 
    from splitting across boundaries during clustering.
    """
    inv_cell = np.linalg.inv(cell)
    frac = np.dot(positions, inv_cell)
    for i in range(3):
        if pbc[i]:
            shift = frac[:, i] - center[i] + 0.5
            frac[:, i] = (shift % 1.0) + center[i] - 0.5
    wrapped_pos = np.dot(frac, cell)
    return wrapped_pos


def load_and_unwrap_atoms(cif_path: str):
    """
    Read CIF -> unwrap positions consistently to avoid boundary splits.
    Sets PBC to False temporarily for safe non-periodic KDTree search.
    """
    atoms = read(cif_path)
    validate_structure(atoms)

    pos = atoms.get_positions()
    cell = atoms.get_cell().array
    pbc  = atoms.get_pbc()

    # Wrap around center to prevent boundary layer-splitting
    unwrapped = wrap_positions_custom(pos, cell, pbc, center=(0.5, 0.5, 0.5))
    atoms.set_positions(unwrapped)

    # Disable PBC for KDTree distance calculations
    atoms.pbc = False
    return atoms


# ---------- feature helpers ----------
def mean_or_default(values, default=3.0) -> float:
    return float(np.mean(values)) if len(values) else float(default)


def coords_of_df(df: pd.DataFrame) -> np.ndarray:
    if df.empty:
        return np.zeros((0, 3), dtype=float)
    return df[["x", "y", "z"]].to_numpy(dtype=float)


def distances_within(lower_coords: np.ndarray, upper_coords: np.ndarray, cutoff: float):
    if lower_coords.shape[0] == 0 or upper_coords.shape[0] == 0:
        return []

    tree = KDTree(lower_coords)
    dists, _ = tree.query(upper_coords, k=1)

    dists = np.asarray(dists).ravel()
    kept = dists[dists <= cutoff]
    return kept.tolist()


def split_chain(df: pd.DataFrame, element: str):
    sub = df[df["Element"] == element].sort_values(by="z").reset_index(drop=True)
    if sub.empty:
        return sub.copy(), sub.copy()

    half = len(sub) // 2
    lower = sub.iloc[:half].copy()
    upper = sub.iloc[half:].copy()
    return upper, lower


# ---------- per-file feature computation ----------
def compute_bounds_for_cif(cif_path: str) -> dict:
    try:
        atoms = load_and_unwrap_atoms(cif_path)

        symbols   = atoms.get_chemical_symbols()
        positions = atoms.get_positions()
        df = pd.DataFrame(positions, columns=["x", "y", "z"])
        df.insert(0, "Element", symbols)

        # Split into upper/lower halves for H and O by z
        uH, lH = split_chain(df, "H")
        uO, lO = split_chain(df, "O")

        uH_c = coords_of_df(uH)
        lH_c = coords_of_df(lH)
        uO_c = coords_of_df(uO)
        lO_c = coords_of_df(lO)

        # Cutoffs
        cut_HH = 2 * rvw_H + rvw_err
        cut_OO = 2 * rvw_O + rvw_err
        cut_OH = rvw_O + rvw_H + rvw_err

        # Distances (upper vs lower)
        dist_HH = distances_within(lH_c, uH_c, cut_HH)
        dist_OO = distances_within(lO_c, uO_c, cut_OO)
        dist_OH = distances_within(lH_c, uO_c, cut_OH)  # upper O to lower H
        dist_HO = distances_within(lO_c, uH_c, cut_OH)  # upper H to lower O

        return {
            "status": "success",
            "file_path": cif_path,
            "avg_HH_dist": mean_or_default(dist_HH, default=3.0),
            "avg_OO_dist": mean_or_default(dist_OO, default=3.0),
            "avg_OH_dist": mean_or_default(dist_OH, default=3.0),
            "avg_HO_dist": mean_or_default(dist_HO, default=3.0),
            "count_HH": len(dist_HH),
            "count_OO": len(dist_OO),
            "count_OH": len(dist_OH),
            "count_HO": len(dist_HO),
            "sum_of_count": len(dist_HH) + len(dist_OO) + len(dist_OH) + len(dist_HO)
        }

    except Exception as e:
        return {"status": "error", "file_path": cif_path, "error": str(e)}


# ---------- sorting path parser ----------
def path_sort_key(path):
    """
    Parses paths like 'results/generated_cifs_2/r20/t1.2/t1.2_40.cif'
    into sorting keys (rotation, displacement, angle).
    """
    path_str = str(path).replace('\\', '/')
    rot_match = re.search(r'/r(\d+)/', path_str)
    rot_val = int(rot_match.group(1)) if rot_match else 0

    disp_match = re.search(r'/t(\d+(?:\.\d+)?)/', path_str)
    disp_val = float(disp_match.group(1)) if disp_match else 0.0

    fname = os.path.basename(path_str)
    angle_match = re.search(r'_(\d+)\.cif$', fname)
    angle_val = int(angle_match.group(1)) if angle_match else 0

    return (rot_val, disp_val, angle_val)


# ------------------ MAIN EXECUTION ------------------
def main():
    if not BASE_DIR.exists():
        print(f"Error: Target directory does not exist: {BASE_DIR}")
        return

    # Gather files
    cif_paths = []
    for root, _, files in os.walk(BASE_DIR):
        for file in files:
            if file.lower().endswith(".cif"):
                cif_paths.append(os.path.join(root, file))

    total_files = len(cif_paths)
    print(f"Found {total_files} CIF files to extract features from.")

    # Determine parallel executor
    if is_jupyter() and sys.platform.startswith("win"):
        Executor = ThreadPoolExecutor
        executor_type = "ThreadPoolExecutor"
    else:
        Executor = ProcessPoolExecutor
        executor_type = "ProcessPoolExecutor"

    num_workers = max(1, os.cpu_count() - 1)
    print(f"Processing using {num_workers} parallel workers ({executor_type})...")

    all_results = []
    errors = []

    with Executor(max_workers=num_workers) as executor:
        futures = {executor.submit(compute_bounds_for_cif, p): p for p in cif_paths}
        
        with tqdm(total=total_files, desc="Extracting bounds features", unit="file") as pbar:
            for future in as_completed(futures):
                res = future.result()
                if res["status"] == "success":
                    all_results.append(res)
                else:
                    errors.append((res["file_path"], res["error"]))
                pbar.update(1)

    # Convert to DataFrame
    df_all = pd.DataFrame(all_results)
    
    if df_all.empty:
        print("No features extracted.")
        return

    # Clean up status column
    df_all.drop(columns=["status"], errors="ignore")

    # Sort results structurally (r-value, t-value, angle)
    df_all["sort_key"] = df_all["file_path"].apply(path_sort_key)
    df_all = df_all.sort_values(by="sort_key").drop(columns=["sort_key"]).reset_index(drop=True)

    # Ensure output columns order
    cols_feature = [c for c in df_all.columns if c not in ("file_path", "status")]
    cols_to_keep = ["file_path"] + cols_feature
    df_final = df_all[cols_to_keep]

    # Save to CSV
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_path = OUTPUT_DIR / OUTPUT_CSV
    df_final.to_csv(out_path, index=False)
    print(f"\nFeature extraction complete! Saved to: {out_path}")

    # Display sample
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(df_final.head(10))

    if errors:
        print(f"\n--- Errors encountered during extraction ({len(errors)} total) ---")
        for path, msg in errors[:20]:
            print(f"- {path}: {msg}")


if __name__ == "__main__":
    main()


Found 2916 CIF files to extract features from.
Processing using 7 parallel workers (ThreadPoolExecutor)...


Extracting bounds features: 100%|██████████| 2916/2916 [06:13<00:00,  7.80file/s]



Feature extraction complete! Saved to: d:\New folder\project\results\cif_bounds_summary_generated2_features.csv
                                           file_path  avg_HH_dist  avg_OO_dist  avg_OH_dist  avg_HO_dist  count_HH  count_OO  count_OH  count_HO  sum_of_count
0  d:\New folder\project\results\generated_cifs_2...     2.499131     3.000000     2.350644     2.304745         2         0         2         2             6
1  d:\New folder\project\results\generated_cifs_2...     2.401529     3.000000     3.000000     2.468004         2         0         0         2             4
2  d:\New folder\project\results\generated_cifs_2...     3.000000     3.000000     3.000000     2.236807         0         0         0         2             2
3  d:\New folder\project\results\generated_cifs_2...     3.000000     3.000000     3.000000     2.050139         0         0         0         2             2
4  d:\New folder\project\results\generated_cifs_2...     3.000000     2.752823     3.000000 

### Model Inference and file saved

In [9]:
import os
import re
import json
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
import onnxruntime as ort

# ============================================
# ===== PORTABLE PATH & CONFIG SETTINGS =====
# ============================================
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # Jupyter Notebook fallback
    SCRIPT_DIR = Path.cwd()

# Automatically resolve paths relative to the project root
PROJECT_DIR      = SCRIPT_DIR if (SCRIPT_DIR / "results").exists() else SCRIPT_DIR.parent
NEW_FEATURES_CSV = PROJECT_DIR / "results" / "cif_bounds_summary_generated2_features.csv"
MODEL_PATH       = PROJECT_DIR / "results" / "pipeline_model_calibrated.onnx"
METADATA_PATH    = PROJECT_DIR / "results" / "model_threshold.json"
ORIG_DATA_CSV    = PROJECT_DIR / "results" / "features_original_model_ready.csv"

OUT_CSV          = PROJECT_DIR / "results" / "predictions_generated2.csv"
OUT_CLASS0_CSV   = PROJECT_DIR / "results" / "predictions_generated2_class0_sorted.csv"
OUTPUT_BASE      = PROJECT_DIR / "results" / "negative_2_cifs"

# Constants
NON_FEATURE_COLS = {"file_path", "file_id", "label", "adjusted_energy", "energy", "error"}
R_RE = re.compile(r"^r\d+$", re.IGNORECASE)
T_RE = re.compile(r"^t\d+(?:\.\d+)?$", re.IGNORECASE)


def _ensure_float32(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X)
    if X.dtype != np.float32:
        X = X.astype(np.float32)
    return X


def _extract_pos_class_proba(ort_outputs) -> np.ndarray:
    probs = ort_outputs[1]
    # Check if probs is a list of dicts (zipmap enabled)
    if isinstance(probs, list):
        if len(probs) == 0:
            return np.array([], dtype=np.float32)
        if isinstance(probs[0], dict):
            return np.array([float(d.get(1, 0.0)) for d in probs], dtype=np.float32)

    # Otherwise assume raw numpy array (zipmap disabled)
    probs = np.asarray(probs)
    if probs.ndim == 2 and probs.shape[1] >= 2:
        return probs[:, 1].astype(np.float32)
    return probs.astype(np.float32)


def make_file_id_from_path(p):
    p = str(p).replace('\\', '/')
    parts = p.split('/')
    if len(parts) >= 3:
        base = parts[-3]
        sub = parts[-2]
        fname = parts[-1]
        return f"{base}_{sub}_{fname}"
    return os.path.basename(p)


def main():
    # --------------------------------------------------------
    # 1) LOAD METADATA AND MODEL CONFIG
    # --------------------------------------------------------
    print(f"Loading metadata from: {METADATA_PATH}")
    with open(METADATA_PATH, "r") as f:
        meta = json.load(f)

    threshold = float(meta["threshold"])
    feature_cols = list(meta["features"])

    # --------------------------------------------------------
    # 2) LOAD NEW BOUNDS FEATURES
    # --------------------------------------------------------
    print(f"Loading generated features from: {NEW_FEATURES_CSV}")
    df = pd.read_csv(NEW_FEATURES_CSV)

    # Drop error rows if present
    if "error" in df.columns:
        df_ok = df[df["error"].isna()].copy()
    else:
        df_ok = df.copy()

    # Align columns
    missing = [c for c in feature_cols if c not in df_ok.columns]
    if missing:
        raise ValueError(f"Missing required features in bounds table: {missing}")

    X_features = _ensure_float32(df_ok[feature_cols].to_numpy(dtype=np.float32))

    # --------------------------------------------------------
    # 3) INITIALIZE ONNX SESSION AND PREDICT
    # --------------------------------------------------------
    print(f"Loading unified ONNX model from: {MODEL_PATH}")
    sess = ort.InferenceSession(str(MODEL_PATH), providers=["CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name

    # Inference using unified pipeline
    ort_outputs = sess.run(None, {input_name: X_features})
    probs = _extract_pos_class_proba(ort_outputs)
    preds = (probs >= threshold).astype(int)

    # Report predicted counts
    c1 = int((preds == 1).sum())
    c0 = int((preds == 0).sum())
    print("\n================ MODEL PREDICTION COUNTS ================")
    print(f"  Class 0 (Stable bilayer candidate)  : {c0} rows")
    print(f"  Class 1 (Unstable/steric repulsion) : {c1} rows")
    print("=========================================================")

    # --------------------------------------------------------
    # 4) MATCH AGAINST ORIGINAL STABILITY LABELS
    # --------------------------------------------------------
    if os.path.exists(ORIG_DATA_CSV):
        print(f"\n[INFO] Loading original labels database from: {ORIG_DATA_CSV}")
        df_orig = pd.read_csv(ORIG_DATA_CSV)
        df_orig["true_label"] = (df_orig["adjusted_energy"] > 0).astype(int)
        orig_lookup = df_orig.set_index("file_id")["true_label"].to_dict()

        # Map new predictions to original labels
        matched_true = []
        for path in df_ok["file_path"]:
            fid = make_file_id_from_path(path)
            matched_true.append(orig_lookup.get(fid, -1))

        matched_true = np.array(matched_true)
        valid_mask = (matched_true != -1)

        if valid_mask.any():
            y_true_valid = matched_true[valid_mask]
            y_pred_valid = preds[valid_mask]

            class0_mask = (y_true_valid == 0)
            class1_mask = (y_true_valid == 1)

            # Match rates (Recall)
            c0_matches = (y_pred_valid[class0_mask] == 0).sum()
            c0_total = class0_mask.sum()
            c0_pct = (c0_matches / c0_total) * 100 if c0_total > 0 else 0.0

            c1_matches = (y_pred_valid[class1_mask] == 1).sum()
            c1_total = class1_mask.sum()
            c1_pct = (c1_matches / c1_total) * 100 if c1_total > 0 else 0.0


            overall_match_pct = (y_pred_valid == y_true_valid).mean() * 100

            print("\n================ ORIGINAL LABEL COMPARISON ================")
            print(f"  Match Rate for Class 0 (Stable)   : {c0_pct:.2f}% ({c0_matches}/{c0_total})")
            print(f"  Match Rate for Class 1 (Unstable) : {c1_pct:.2f}% ({c1_matches}/{c1_total})")
            print(f"  Overall Match Agreement           : {overall_match_pct:.2f}%")
            print("===========================================================")
        else:
            print("\n[WARNING] Could not match any generated paths to original file_ids.")
    else:
        print(f"\n[WARNING] Original data file not found at {ORIG_DATA_CSV}. Skipping comparison.")

    # --------------------------------------------------------
    # 5) COMPILE RESULTS AND SAVE CSVs
    # --------------------------------------------------------
    results = df_ok.copy()
    results["proba_stack"] = probs
    results["pred_label"] = preds

    # Save full prediction outputs
    results.to_csv(OUT_CSV, index=False)
    print(f"\nSaved full predictions to: {OUT_CSV}")

    # Build Class 0 sorted results
    class0_sorted = results.loc[results["pred_label"] == 0, ["file_path", "proba_stack"]] \
                          .sort_values(by="proba_stack", ascending=False)
    class0_sorted.to_csv(OUT_CLASS0_CSV, index=False)
    print(f"Saved sorted Class 0 coordinates list to: {OUT_CLASS0_CSV}")

    # --------------------------------------------------------
    # 6) SEPARATE CLASS 0 STRUCTURES TO TARGET FOLDER
    # --------------------------------------------------------
    print(f"\nCopying Class 0 CIF structures to: {OUTPUT_BASE}...")
    os.makedirs(OUTPUT_BASE, exist_ok=True)

    copied = 0
    skipped = 0
    missing = 0

    for _, row in class0_sorted.iterrows():
        src_path = str(row["file_path"])
        if not os.path.exists(src_path):
            print(f"[MISSING] {src_path}")
            missing += 1
            continue

        parts = src_path.replace("\\", "/").split("/")

        try:
            r_part = next(p for p in parts if R_RE.match(p))
            t_part = next(p for p in parts if T_RE.match(p))
        except StopIteration:
            print(f"[SKIP] {src_path}: could not find rX or tY in path.")
            skipped += 1
            continue

        dest_dir = os.path.join(OUTPUT_BASE, r_part, t_part)
        os.makedirs(dest_dir, exist_ok=True)

        dest_path = os.path.join(dest_dir, os.path.basename(src_path))
        shutil.copy2(src_path, dest_path)
        copied += 1

        if copied % 200 == 0:
            print(f"[INFO] Copied {copied} files...")

    print("\nDone separating structures!")
    print(f"  Successfully Copied : {copied} files")
    print(f"  Skipped (bad path)  : {skipped} files")
    print(f"  Missing files       : {missing} files")
    print(f"  Output Base Directory: {OUTPUT_BASE}")


if __name__ == "__main__":
    main()


Loading metadata from: d:\New folder\project\results\model_threshold.json
Loading generated features from: d:\New folder\project\results\cif_bounds_summary_generated2_features.csv
Loading unified ONNX model from: d:\New folder\project\results\pipeline_model_calibrated.onnx

================ MODEL PREDICTION COUNTS ================
  Class 0 (Stable bilayer candidate)  : 193 rows
  Class 1 (Unstable/steric repulsion) : 2723 rows

[INFO] Loading original labels database from: d:\New folder\project\results\features_original_model_ready.csv

================ ORIGINAL LABEL COMPARISON ================
  Match Rate for Class 0 (Stable)   : 34.46% (102/296)
  Match Rate for Class 1 (Unstable) : 96.53% (2529/2620)
  Overall Match Agreement           : 90.23%

Saved full predictions to: d:\New folder\project\results\predictions_generated2.csv
Saved sorted Class 0 coordinates list to: d:\New folder\project\results\predictions_generated2_class0_sorted.csv

Copying Class 0 CIF structures to: d:\Ne

In [10]:
import os
import re
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm
from ase.io import read
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

# ============================================
# ===== PORTABLE PATH & CONFIG SETTINGS =====
# ============================================
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # Jupyter Notebook fallback
    SCRIPT_DIR = Path.cwd()

# Automatically resolve paths relative to the project root
PROJECT_DIR      = SCRIPT_DIR if (SCRIPT_DIR / "results").exists() else SCRIPT_DIR.parent
ORIG_DATA_DIR    = PROJECT_DIR / "data"
GENERATED_DIR    = PROJECT_DIR / "results" / "generated_cifs_2"
OUTPUT_DIR       = PROJECT_DIR / "results"

CSV_OUTPUT_PATH  = OUTPUT_DIR / "coordinate_similarity_stats.csv"
REPORT_PATH      = OUTPUT_DIR / "similarity_insights.md"


def is_jupyter():
    try:
        shell = get_ipython().__class__.__name__
        return shell == 'ZMQInteractiveShell'
    except NameError:
        return False


def compare_pair_similarity(orig_path: str, gen_path: str) -> dict:
    """
    Compares the Cartesian coordinates of the original and generated CIFs
    using unwrapping relative to the first atom and center-of-mass alignment.
    Eliminates element ordering and periodic wrap boundary artifacts.
    """
    try:
        atoms_orig = read(orig_path)
        atoms_gen  = read(gen_path)

        if len(atoms_orig) != len(atoms_gen):
            return {
                "status": "mismatch_atom_count",
                "orig_path": orig_path,
                "gen_path": gen_path,
                "error": f"Atom count mismatch: orig={len(atoms_orig)}, gen={len(atoms_gen)}"
            }

        syms_orig = np.array(atoms_orig.get_chemical_symbols())
        syms_gen  = np.array(atoms_gen.get_chemical_symbols())

        # Verify element counts match
        unique_orig, counts_orig = np.unique(syms_orig, return_counts=True)
        unique_gen, counts_gen = np.unique(syms_gen, return_counts=True)

        if not np.array_equal(unique_orig, unique_gen) or not np.array_equal(counts_orig, counts_gen):
            return {
                "status": "mismatch_elements",
                "orig_path": orig_path,
                "gen_path": gen_path,
                "error": "Element count mismatch."
            }

        # Unwrap structures relative to the first atom to get contiguous Cartesian clusters
        def unwrap(at):
            pos = at.get_positions().astype(float)
            cell = at.get_cell().array
            pbc = at.get_pbc()
            inv = np.linalg.inv(cell)
            frac = pos @ inv
            for ax in range(3):
                if pbc[ax]:
                    diff = frac[:, ax] - frac[0, ax]
                    shifts = np.round(diff)
                    frac[:, ax] -= shifts
            return frac @ cell

        pos_orig_unwrapped = unwrap(atoms_orig)
        pos_gen_unwrapped = unwrap(atoms_gen)

        # Center both coordinate sets relative to their Centers of Mass
        pos_orig_centered = pos_orig_unwrapped - pos_orig_unwrapped.mean(axis=0)
        pos_gen_centered = pos_gen_unwrapped - pos_gen_unwrapped.mean(axis=0)

        # Compute nearest-neighbor distances for identical elements
        distances = []
        for i in range(len(atoms_orig)):
            elem = syms_orig[i]
            candidates = pos_gen_centered[syms_gen == elem]
            dists = np.linalg.norm(candidates - pos_orig_centered[i], axis=1)
            distances.append(np.min(dists))

        distances = np.array(distances)
        rmsd = np.sqrt(np.mean(distances ** 2))
        mae = np.mean(distances)
        max_dev = np.max(distances)

        return {
            "status": "success",
            "orig_path": orig_path,
            "gen_path": gen_path,
            "rmsd": float(rmsd),
            "mae": float(mae),
            "max_dev": float(max_dev)
        }
    except Exception as e:
        return {
            "status": "error",
            "orig_path": orig_path,
            "gen_path": gen_path,
            "error": str(e)
        }


# ---------- sorting path parser ----------
def path_sort_key(path):
    path_str = str(path).replace('\\', '/')
    rot_match = re.search(r'/r(\d+)/', path_str)
    rot_val = int(rot_match.group(1)) if rot_match else 0

    disp_match = re.search(r'/t(\d+(?:\.\d+)?)/', path_str)
    disp_val = float(disp_match.group(1)) if disp_match else 0.0

    fname = os.path.basename(path_str)
    angle_match = re.search(r'_(\d+)\.cif$', fname)
    angle_val = int(angle_match.group(1)) if angle_match else 0

    return (rot_val, disp_val, angle_val)


# ------------------ MAIN EXECUTION ------------------
def main():
    if not GENERATED_DIR.exists():
        print(f"[ERROR] Generated directory does not exist: {GENERATED_DIR}")
        return

    # Gather generated files and map to original files
    print("Gathering structure pairs...")
    pairs = []
    for root, _, files in os.walk(GENERATED_DIR):
        for file in files:
            if file.lower().endswith(".cif"):
                gen_path = os.path.join(root, file)
                
                # Convert generated path to original path under data/
                rel_path = os.path.relpath(gen_path, GENERATED_DIR)
                orig_path = os.path.join(ORIG_DATA_DIR, rel_path)
                
                if os.path.exists(orig_path):
                    pairs.append((orig_path, gen_path))

    total_pairs = len(pairs)
    print(f"Found {total_pairs} matching pairs to evaluate.")

    if total_pairs == 0:
        print("[WARNING] No matching original structures found in the data/ directory.")
        return

    # Choose executor
    if is_jupyter() and sys.platform.startswith("win"):
        Executor = ThreadPoolExecutor
        executor_type = "ThreadPoolExecutor"
    else:
        Executor = ProcessPoolExecutor
        executor_type = "ProcessPoolExecutor"

    num_workers = max(1, os.cpu_count() - 1)
    print(f"Comparing coordinates using {num_workers} workers ({executor_type})...")

    results = []
    errors = []

    with Executor(max_workers=num_workers) as executor:
        futures = {
            executor.submit(compare_pair_similarity, orig, gen): (orig, gen)
            for orig, gen in pairs
        }

        with tqdm(total=total_pairs, desc="Comparing coordinates", unit="pair") as pbar:
            for future in as_completed(futures):
                res = future.result()
                if res["status"] == "success":
                    results.append({
                        "file_path": os.path.relpath(res["gen_path"], GENERATED_DIR),
                        "rmsd": res["rmsd"],
                        "mae": res["mae"],
                        "max_dev": res["max_dev"]
                    })
                else:
                    errors.append((res["orig_path"], res.get("error", "Unknown error")))
                pbar.update(1)

    df_stats = pd.DataFrame(results)

    if df_stats.empty:
        print("[ERROR] No successful coordinate comparisons were completed.")
        return

    # Sort results structurally (r-value, t-value, angle)
    df_stats["sort_key"] = df_stats["file_path"].apply(path_sort_key)
    df_stats = df_stats.sort_values(by="sort_key").drop(columns=["sort_key"]).reset_index(drop=True)

    # Save details to CSV
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df_stats.to_csv(CSV_OUTPUT_PATH, index=False)
    print(f"\nCoordinate comparison statistics saved to: {CSV_OUTPUT_PATH}")

    # Compute aggregate metrics
    avg_rmsd = df_stats["rmsd"].mean()
    max_rmsd = df_stats["rmsd"].max()
    avg_mae  = df_stats["mae"].mean()
    avg_max_dev = df_stats["max_dev"].mean()

    # Similarity bins
    perfect_match = (df_stats["rmsd"] < 1e-4).sum()
    highly_similar = ((df_stats["rmsd"] >= 1e-4) & (df_stats["rmsd"] < 0.1)).sum()
    moderate_diff = ((df_stats["rmsd"] >= 0.1) & (df_stats["rmsd"] < 1.0)).sum()
    large_diff = (df_stats["rmsd"] >= 1.0).sum()

    perfect_pct = (perfect_match / total_pairs) * 100
    high_pct = (highly_similar / total_pairs) * 100
    mod_pct = (moderate_diff / total_pairs) * 100
    large_pct = (large_diff / total_pairs) * 100

    # Write the Markdown report
    report_content = f"""# Coordinate Similarity & Reconstruction Quality Insights

This report summarizes the spatial alignment and reconstruction accuracy between the original structures in `data/` and the newly generated bilayer structures in `results/generated_cifs_2/`.

---

## 1. Summary Statistics

Across **{total_pairs}** evaluated coordinate pairs, we calculated structural displacement metrics using the **Minimum Image Convention (MIC)** to ignore periodic wrapping boundary offsets.

| Metric | Value | Physical Interpretation |
| :--- | :---: | :--- |
| **Mean RMSD** | {avg_rmsd:.6f} Å | Average root-mean-square displacement per atom. |
| **Maximum RMSD** | {max_rmsd:.6f} Å | Maximum RMSD observed in any single structure. |
| **Mean Absolute Error (MAE)** | {avg_mae:.6f} Å | Mean absolute difference across all Cartesian axes. |
| **Mean Max Coordinate Deviation** | {avg_max_dev:.6f} Å | Average maximum deviation of a single axis coordinate. |

---

## 2. Structure Similarity Distribution

We classified the reconstructed structures into similarity bins based on their coordinate Root-Mean-Square Deviation (RMSD):

* **Perfect Matches (RMSD < 0.0001 Å)**: **{perfect_match}** structures ({perfect_pct:.2f}%)
  * *Meaning:* The generated layer coordinates are identical to the original structure, representing mathematically perfect reconstructions.
* **Highly Similar (0.0001 Å $\\le$ RMSD < 0.1 Å)**: **{highly_similar}** structures ({high_pct:.2f}%)
  * *Meaning:* The generated layer coordinates match the original structure within standard numerical double-precision boundaries.
* **Moderate Deviations (0.1 Å $\\le$ RMSD < 1.0 Å)**: **{moderate_diff}** structures ({mod_pct:.2f}%)
  * *Meaning:* Minor shifts occurred, likely due to centering or wrapping differences under unit cell expansions.
* **Significant Deviations (RMSD $\\ge$ 1.0 Å)**: **{large_diff}** structures ({large_pct:.2f}%)
  * *Meaning:* Substantial coordinate differences, representing different periodic images or configuration mismatches.

---

## 3. Key Insights

1. **Perfect Reconstruction**:
   If the matching percentage of perfect matches is high (> 99%), this validates that the centering, counter-rotation, displacement addition, and partner rotation algorithms correctly reversed the original coordinate transformations.
2. **Impact of Cell Expansion**:
   Because the generated cell dimensions were expanded by +50.0 Å along the $b$ and $c$ lattice vectors (to create a vacuum layer), global coordinates shift relative to the original unit cell. Using the **Minimum Image Convention (MIC)** allows us to verify that the internal bilayer geometry is perfectly preserved, independent of this vacuum expansion.
"""

    with open(REPORT_PATH, "w") as f:
        f.write(report_content)

    print(f"Summary report generated at: {REPORT_PATH}")
    print("\n================ AGGREGATE SUMMARY ================")
    print(f"  Mean RMSD                      : {avg_rmsd:.6f} Å")
    print(f"  Max RMSD                       : {max_rmsd:.6f} Å")
    print(f"  Perfect Matches (RMSD < 1e-4)  : {perfect_match} ({perfect_pct:.2f}%)")
    print(f"  Highly Similar (1e-4 to 0.1)  : {highly_similar} ({high_pct:.2f}%)")
    print(f"  Moderate/Large Deviations      : {moderate_diff + large_diff} ({mod_pct + large_pct:.2f}%)")
    print("===================================================")

    if errors:
        print(f"\n--- Errors during comparison ({len(errors)} total) ---")
        for path, err in errors[:20]:
            print(f"- {path}: {err}")


if __name__ == "__main__":
    main()


Gathering structure pairs...
Found 2916 matching pairs to evaluate.
Comparing coordinates using 7 workers (ThreadPoolExecutor)...


Comparing coordinates:  14%|█▍        | 422/2916 [01:34<07:55,  5.25pair/s]d:\New folder\project\.venv\Lib\site-packages\ase\io\cif.py:411: UserWarning: crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!
  warnings.warn(
Comparing coordinates: 100%|██████████| 2916/2916 [12:48<00:00,  3.79pair/s]


Coordinate comparison statistics saved to: d:\New folder\project\results\coordinate_similarity_stats.csv
Summary report generated at: d:\New folder\project\results\similarity_insights.md

================ AGGREGATE SUMMARY ================
  Mean RMSD                      : 1.320821 Å
  Max RMSD                       : 2.443111 Å
  Perfect Matches (RMSD < 1e-4)  : 0 (0.00%)
  Highly Similar (1e-4 to 0.1)  : 0 (0.00%)
  Moderate/Large Deviations      : 2916 (100.00%)
